In [ ]:
#Imports
from PIL import Image
import pandas as pd
from pathlib import Path
import re
import os
import numpy as np
from matplotlib import pyplot as plt

In [ ]:
def scan_dir(directory, file_paths):
    for entry in os.scandir(directory):
        if entry.is_dir():
            scan_dir(entry.path, file_paths)
        elif entry.is_file() and ".ods" not in entry.name and ".yaml" not in entry.name:
            file_paths.append(entry.path)
   
    return file_paths

In [ ]:
directory = "C:\\Users\\Installer\\Documents\\Data Science Class\\Computer-Vision-Pipeline\\data\\license_plate_detection"

file_paths = []
file_paths = scan_dir(directory, file_paths)

txt_paths = []
jpg_paths = []
names = []
for file in file_paths:
    if '.txt' in file:
        txt_paths.append(file)
    elif '.jpg' in file:
        jpg_paths.append(file)
        splits = re.split(r'\\|\.', file)
        with Image.open(file) as img:
                width, height = img.size
                name = {
                     "name": splits[-2],
                     "width": int(width),
                     "height": int(height)
                }
        names.append(name)

In [ ]:
#Creates the dataframe for the bounding boxes in each image
crops = []
name_lookup = {n["name"]: n for n in names}

for file in txt_paths:
    stem = Path(file).stem

    if stem not in name_lookup:
        continue

    name = name_lookup[stem]
    with Path.open(file) as f:
        next(f) #Skips first line in file
        for line in f:
            line_split = line.split()
            crop = {
                "name": name["name"],
                "class": int(line_split[0]),
                "X Center": float(line_split[1]) * name["width"],
                "Y Center": float(line_split[2]) * name["height"],
                "Width": float(line_split[3]) * name["width"],
                "Height": float(line_split[4]) * name["height"]
            }
            crops.append(crop)

df = pd.DataFrame(crops)
df["x min"] = df["X Center"] - df["Width"]/2
df["y min"] = df["Y Center"] - df["Height"]/2
df["x max"] = df["X Center"] + df["Width"]/2
df["y max"] = df["Y Center"] + df["Height"]/2
df = df.drop(columns=["X Center", "Y Center", "Width", "Height"])

df

In [ ]:
# Function to create image crops locally
def make_crops(jpg_path):
    name = Path(jpg_path).stem
    for _, row in df.iterrows():
        if name == row["name"]:
            output_path = f'crops/{name}_crop.jpg'
            folder_path = os.path.dirname(output_path)
            if folder_path and not os.path.exists(folder_path):
                os.makedirs(folder_path, exist_ok=True)
                
            with Image.open(jpg_path) as img:
                img_crop = img.crop((row["x min"],row["y min"],row["x max"], row["y max"]))
                img_crop.save(output_path, format="JPEG")
            
            print(f"Saved {name} to {output_path}")

In [ ]:
for path in jpg_paths[:50]:
    make_crops(path)